# Deterministic Problem Generator

This notebook builds a LeetCode-style problem package with minimal LLM use.

Pipeline:
1. Configure idea and toggles
2. Generate `question.md` with LLM
3. Generate deterministic testcases
4. Generate language driver/signature with LLM
5. Run python tests automatically
6. Emit metadata and final artifact summary

In [22]:
# Cell 1 - User Configuration & Prompts
from pathlib import Path

# --- CORE SETTINGS ---
IDEA = "Smart Order Router minimizing trading cost across exchanges"
LANGUAGES = ["python", "cpp", "java"]
DIFFICULTY = "medium"
DATA_STRUCTURE = "sorting"

GENERATE_QUESTION = True
GENERATE_TESTCASES = True
GENERATE_DRIVER = True
RUN_TESTS = True

OUTPUTS_ROOT = Path("data/questions")
RANDOM_SEED = 42

# --- PROMPT TEMPLATES ---

QUESTION_PROMPT_TEMPLATE = """
Generate a LeetCode-style algorithm problem in markdown.

Idea: {IDEA}
Difficulty: {DIFFICULTY}
Data Structure Focus: {DATA_STRUCTURE}

Rules:
- no solution
- include problem statement
- include input/output sections
- DO NOT include a function signature section (it will be provided separately)
- include at least 2 examples
- include constraints
- include 2-4 hints that do not reveal the full solution
""".strip()

# New: One request for signature, one for driver
SIGNATURE_PROMPT_TEMPLATE = """
Generate ONLY the function signature/stub for the following problem.
Language: {lang}
File extension: {ext}
Problem Idea: {IDEA}

Instructions:
- Provide ONLY the code.
- Ensure the function name is 'minimumCost'.
- Include necessary imports if required for the signature.
- Do not include any explanation or driver logic.
""".strip()

DRIVER_PROMPT_TEMPLATE = """
Generate ONLY the driver code for the following problem.
Language: {lang}
File extension: {ext}
Problem Idea: {IDEA}

Requirements:
- read testcase path from argv[1], default to testcases.txt.
- testcase line format: orderQty n p1 q1 p2 q2 ... expected.
- parse each line.
- call minimumCost(orderQty, exchanges).
- compare actual to expected.
- print JSON array with objects: {{"output": bool, "case": int, "expected": any, "result": any}}.
- For python specifically: imports 'from signature import minimumCost'.
- For others: include external declaration of minimumCost.
- Do not include any solution implementation or signature stub.
- Provide ONLY the code.
""".strip()

# --- ORACLE IMPLEMENTATION ---
ORACLE_PYTHON_CODE = '''
def minimumCost(orderQty, exchanges):
    indexed = [(price, idx, qty) for idx, (price, qty) in enumerate(exchanges)]
    indexed.sort(key=lambda x: (x[0], x[1]))

    remaining = orderQty
    total = 0
    for price, _idx, qty in indexed:
        if remaining == 0:
            break
        buy = min(remaining, qty)
        total += buy * price
        remaining -= buy

    return total if remaining == 0 else -1
'''

print("Configuration updated. Requests are now broken down.")

Configuration updated. Requests are now broken down.


In [ ]:
# Cell 2 - LLM & Model Selection
import json
import requests
import litellm

API_BASE = "http://localhost:1234/v1"
litellm.api_base = API_BASE
litellm.api_key = "lm-studio"
TEMPERATURE = 0

def get_available_models():
    try:
        # Request only 1 row or limit fields if possible, but standard OpenAI /models doesn't support it.
        # We manually truncate the list if it's too long to save context.
        resp = requests.get(f"{API_BASE}/models", timeout=5)
        if resp.status_code == 200:
            models = [m['id'] for m in resp.json().get('data', [])]
            return models[:10] # Keep only top 10 models to avoid context bloat
    except:
        pass
    return ["openai/local-model"]

available_models = get_available_models()
# Only print the summary to keep notebook output small
print(f"Found {len(available_models)} models. Top: {available_models[:3]}")

MODEL = "openai/local-model" 
if available_models:
    first_model = available_models[0]
    MODEL = f"openai/{first_model}" if not first_model.startswith("openai/") else first_model

print(f"Using MODEL: {MODEL}")

def llm_complete(prompt: str) -> str:
    response = litellm.completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
    )
    return response.choices[0].message.content.strip()

Found 5 models. Top: ['zai-org/glm-4.6v-flash', 'liquid/lfm2.5-1.2b', 'qwen/qwen3-8b']
Using MODEL: openai/zai-org/glm-4.6v-flash


In [24]:
# Cell 3 - Slug & Directory Setup
import re

def to_slug(text: str) -> str:
    # Use hyphens for slugs to match repo pattern
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "-", text.strip().lower())
    cleaned = re.sub(r"-+", "-", cleaned).strip("-")
    return cleaned or "untitled-problem"

problem_slug = to_slug(IDEA)
problem_dir = OUTPUTS_ROOT / problem_slug
problem_dir.mkdir(parents=True, exist_ok=True)

selected_languages = [str(lang).strip().lower() for lang in LANGUAGES if str(lang).strip()]
language_driver_dirs = {lang: (problem_dir / "drivers" / lang) for lang in selected_languages}

for d in language_driver_dirs.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"Problem: {problem_slug}")
print(f"Path: {problem_dir.resolve()}")

Problem: smart-order-router-minimizing-trading-cost-across-exchanges
Path: /Users/shivamjain/Development/intui/data/questions/smart-order-router-minimizing-trading-cost-across-exchanges


In [25]:
# Cell 4 - Question Generation (LLM)
question_path = problem_dir / "Question.md"

if GENERATE_QUESTION:
    prompt = QUESTION_PROMPT_TEMPLATE.format(
        IDEA=IDEA, 
        DIFFICULTY=DIFFICULTY, 
        DATA_STRUCTURE=DATA_STRUCTURE
    )

    try:
        question_md = llm_complete(prompt)
    except Exception as e:
        print("Fallback used:", e)
        question_md = f"# {IDEA}\n\n## Problem Statement\n{IDEA}"

    question_path.write_text(question_md, encoding="utf-8")
    print("Wrote", question_path)
else:
    print("Skipped.")

Wrote data/questions/smart-order-router-minimizing-trading-cost-across-exchanges/Question.md


In [26]:
# Cell 5 - Testcase Generator Creation (Deterministic)
import textwrap


testcase_generator_path = problem_dir / "testcase_generator.py"

def build_testcase_generator_script(oracle_code: str, seed: int = 42) -> str:
    script = f'''import random\n\n{oracle_code}\n\nrandom.seed({seed})\n\n# Line format:\n# orderQty n p1 q1 p2 q2 ... expected\n\ndef _serialize(order_qty, exchanges, expected):\n    flat = []\n    for p, q in exchanges:\n        flat.append(str(p))\n        flat.append(str(q))\n    return f"{{order_qty}} {{len(exchanges)}} {{' '.join(flat)}} {{expected}}"\n\n\ndef make_cases():\n    cases = []\n\n    # Simple fixed cases\n    cases.append((7, [[10, 5], [8, 4], [9, 3]]))\n    cases.append((10, [[7, 3], [8, 2], [10, 4]]))\n\n    # Edge cases\n    cases.append((1, [[1, 1]]))\n    cases.append((5, [[5, 5]]))\n    cases.append((6, [[2, 1], [2, 1], [2, 1], [2, 1], [2, 1], [2, 1]]))\n\n    # Random medium cases\n    for _ in range(25):\n        n = random.randint(1, 30)\n        order_qty = random.randint(1, 120)\n        exchanges = []\n        for _ in range(n):\n            price = random.randint(1, 100)\n            qty = random.randint(1, 100)\n            exchanges.append([price, qty])\n        cases.append((order_qty, exchanges))\n\n    return cases\n\n\ndef make_submission_cases():\n    cases = []\n\n    # Large deterministic random cases\n    for _ in range(10):\n        n = random.randint(1000, 5000)\n        order_qty = random.randint(1, 10**8)\n        exchanges = []\n        for _ in range(n):\n            price = random.randint(1, 10**6)\n            qty = random.randint(1, 10**6)\n            exchanges.append([price, qty])\n        cases.append((order_qty, exchanges))\n\n    return cases\n\n\ndef write_cases(path, cases):\n    with open(path, 'w', encoding='utf-8') as f:\n        for order_qty, exchanges in cases:\n            expected = minimumCost(order_qty, exchanges)\n            f.write(_serialize(order_qty, exchanges, expected) + '\\n')\n\n\nif __name__ == '__main__':\n    write_cases('testcases.txt', make_cases())\n    write_cases('testcases_submission.txt', make_submission_cases())\n    print('Generated testcases.txt and testcases_submission.txt')\n'''
    return script

if GENERATE_TESTCASES:
    testcase_generator_path.write_text(build_testcase_generator_script(ORACLE_PYTHON_CODE, RANDOM_SEED), encoding="utf-8")
    print("Wrote", testcase_generator_path)
else:
    print("Skipped testcase generator creation by config flag.")

Wrote data/questions/smart-order-router-minimizing-trading-cost-across-exchanges/testcase_generator.py


In [27]:
# Cell 6 - Execute Testcase Generator
import subprocess


def preview_lines(path: Path, limit: int = 5):
    if not path.exists():
        print(f"Missing file: {path}")
        return
    print(f"Preview of {path.name}:")
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= limit:
                break
            print(line.rstrip())

if GENERATE_TESTCASES:
    cmd = ["python3", str(testcase_generator_path.name)]
    result = subprocess.run(cmd, cwd=problem_dir, capture_output=True, text=True)
    print("stdout:", result.stdout.strip())
    if result.stderr.strip():
        print("stderr:", result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"testcase_generator.py failed with code {result.returncode}")

preview_lines(problem_dir / "testcases.txt", 5)
preview_lines(problem_dir / "testcases_submission.txt", 5)

stdout: Generated testcases.txt and testcases_submission.txt
Preview of testcases.txt:
7 3 10 5 8 4 9 3 59
10 3 7 3 8 2 10 4 -1
1 1 1 1 1
5 1 5 5 25
6 6 2 1 2 1 2 1 2 1 2 1 2 1 12
Preview of testcases_submission.txt:
2314352 4247 97574 308529 232361 424046 725392 255124 321081 696213 609806 386946 496250 580359 556712 360456 446174 782170 577125 346860 368901 736997 475787 284077 321518 263616 241737 126517 756322 201940 330877 125363 778968 561895 997068 799214 723689 194144 200846 226896 774484 507720 289931 759783 618227 797136 550122 625781 296746 105410 873075 203548 310637 238534 378412 188159 316951 14836 742412 560082 132732 287637 47727 57175 580239 306328 731296 989950 132414 668855 910648 789360 514724 107571 915113 12862 601950 298152 492199 501969 461866 357257 193320 53873 264743 986777 903583 500936 119630 862051 68518 420173 515634 77681 605051 660022 719797 56214 159098 156446 850545 590181 995231 318595 89319 260248 124206 585182 801578 436391 635771 625085 829152 648

In [28]:
# Cell 7 - Driver & Signature Generation (Sequential LLM Requests)
import re

LANG_EXT = {"python": "py", "cpp": "cpp", "java": "java"}
generated_driver_paths = {}
generated_signature_paths = {}

if GENERATE_DRIVER:
    for lang in selected_languages:
        ext = LANG_EXT[lang]
        lang_dir = language_driver_dirs[lang]
        
        # 1. Signature Request
        s_prompt = SIGNATURE_PROMPT_TEMPLATE.format(lang=lang, ext=ext, IDEA=IDEA)
        print(f"Generating signature for {lang}...")
        try:
            signature_code = llm_complete(s_prompt)
            # Remove possible markdown code blocks if the LLM adds them despite instructions
            signature_code = re.sub(r"```[a-z]*\n|```", "", signature_code).strip()
        except Exception as e:
            print(f"Signature fallback for {lang}: {e}")
            signature_code = "def minimumCost(orderQty, exchanges): pass" if lang == "python" else "// stub"

        # 2. Driver Request
        d_prompt = DRIVER_PROMPT_TEMPLATE.format(lang=lang, ext=ext, IDEA=IDEA)
        print(f"Generating driver for {lang}...")
        try:
            driver_code = llm_complete(d_prompt)
            driver_code = re.sub(r"```[a-z]*\n|```", "", driver_code).strip()
        except Exception as e:
            print(f"Driver fallback for {lang}: {e}")
            driver_code = "# Minimal fallback driver..."

        s_path = lang_dir / f"signature.{ext}"
        d_path = lang_dir / f"driver.{ext}"
        
        s_path.write_text(signature_code + "\n", encoding="utf-8")
        d_path.write_text(driver_code + "\n", encoding="utf-8")
        
        generated_signature_paths[lang] = s_path
        generated_driver_paths[lang] = d_path
        print(f"Finished {lang} files.")
else:
    print("Skipped.")

Generating signature for python...
Generating driver for python...
Finished python files.
Generating signature for cpp...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

Signature fallback for cpp: litellm.BadRequestError: OpenAIException - Error code: 400 - {'error': 'The model has crashed without additional information. (Exit code: null)'}
Generating driver for cpp...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

Driver fallback for cpp: litellm.BadRequestError: OpenAIException - Error code: 400 - {'error': 'Model unloaded.'}


FileNotFoundError: [Errno 2] No such file or directory: 'data/questions/smart-order-router-minimizing-trading-cost-across-exchanges/drivers/cpp/signature.cpp'

In [ ]:
# Cell 8 - Metadata Generation
metadata_path = problem_dir / "metadata.json"

def metadata_for_problem():
    title = IDEA.strip().title()
    return {
        "title": title,
        "difficulty": DIFFICULTY,
        "tags": ["Greedy", "Sorting", "Simulation"],
        "hints": [
            "Try ordering choices so each local decision is globally sensible.",
            "Track remaining quantity after each exchange decision.",
            "Think about an early stop condition once the order is filled.",
        ],
        "learning_objectives": [
            "Apply greedy selection with sorted inputs.",
            "Design deterministic testcase generation.",
            "Build language-specific testing drivers.",
        ],
        "languages": selected_languages,
        "data_structure": DATA_STRUCTURE,
        "slug": problem_slug,
    }

metadata_path.write_text(json.dumps(metadata_for_problem(), indent=2), encoding="utf-8")
print("Wrote", metadata_path)

Wrote data/questions/smart_order_router_minimizing_trading_cost_across_exchanges/metadata.json


In [ ]:
# Cell 9 - Test Runner
import tempfile

if RUN_TESTS:
    if "python" not in selected_languages:
        print("Auto test run in notebook is currently implemented for python only. Add 'python' to LANGUAGES to run local tests.")
    elif "python" not in generated_driver_paths or "python" not in generated_signature_paths:
        print("Python driver/signature not generated yet. Run Cell 7 first.")
    else:
        py_driver_path = generated_driver_paths["python"]
        with tempfile.TemporaryDirectory() as td:
            td_path = Path(td)
            (td_path / "driver.py").write_text(py_driver_path.read_text(encoding="utf-8"), encoding="utf-8")

            # Provide executable signature implementation for local verification.
            impl_code = ORACLE_PYTHON_CODE.strip() + "\n"
            (td_path / "signature.py").write_text(impl_code, encoding="utf-8")

            (td_path / "testcases.txt").write_text((problem_dir / "testcases.txt").read_text(encoding="utf-8"), encoding="utf-8")
            (td_path / "testcases_submission.txt").write_text((problem_dir / "testcases_submission.txt").read_text(encoding="utf-8"), encoding="utf-8")

            cmd = ["python3", "driver.py", "testcases.txt"]
            result = subprocess.run(cmd, cwd=td_path, capture_output=True, text=True)
            print("Driver exit code:", result.returncode)
            if result.stderr.strip():
                print("stderr:", result.stderr.strip())

            if result.returncode == 0:
                parsed = json.loads(result.stdout)
                passed = sum(1 for item in parsed if item.get("output"))
                total = len(parsed)
                print(f"Passed {passed}/{total} local public tests")
            else:
                print("Driver output:")
                print(result.stdout)
else:
    print("Skipped test execution by config flag.")

Driver exit code: 0
Passed 30/30 local public tests


In [ ]:
# Cell 10 - Final Output Summary
print("Problem generated successfully")
print("Question.md:", (problem_dir / "Question.md").resolve())
print("testcase_generator.py:", (problem_dir / "testcase_generator.py").resolve())
print("testcases.txt:", (problem_dir / "testcases.txt").resolve())
print("testcases_submission.txt:", (problem_dir / "testcases_submission.txt").resolve())

if 'generated_signature_paths' in globals() and generated_signature_paths:
    for lang, p in generated_signature_paths.items():
        print(f"signature[{lang}] :", p.resolve())
if 'generated_driver_paths' in globals() and generated_driver_paths:
    for lang, p in generated_driver_paths.items():
        print(f"driver[{lang}] :", p.resolve())

print("metadata.json:", (problem_dir / "metadata.json").resolve())

Problem generated successfully
Question.md: /Users/shivamjain/Development/intui/data/questions/smart_order_router_minimizing_trading_cost_across_exchanges/Question.md
testcase_generator.py: /Users/shivamjain/Development/intui/data/questions/smart_order_router_minimizing_trading_cost_across_exchanges/testcase_generator.py
testcases.txt: /Users/shivamjain/Development/intui/data/questions/smart_order_router_minimizing_trading_cost_across_exchanges/testcases.txt
testcases_submission.txt: /Users/shivamjain/Development/intui/data/questions/smart_order_router_minimizing_trading_cost_across_exchanges/testcases_submission.txt
signature[python] : /Users/shivamjain/Development/intui/data/questions/smart_order_router_minimizing_trading_cost_across_exchanges/drivers/python/signature.py
signature[cpp] : /Users/shivamjain/Development/intui/data/questions/smart_order_router_minimizing_trading_cost_across_exchanges/drivers/cpp/signature.cpp
signature[java] : /Users/shivamjain/Development/intui/data/que